In [21]:
import sqlite3
import pandas as pd

# Carregando a base para treino 

conn = sqlite3.connect("..\data\credit_risk.db")
df= pd.read_sql("SELECT * from credito_integrado",conn)
print(df.dtypes)
print(df.head)

person_age                      int64
person_income                 float64
person_home_ownership             str
person_emp_length              object
loan_intent                       str
loan_grade                        str
loan_amnt                     float64
loan_int_rate                  object
loan_status                     int64
loan_percent_income           float64
cb_person_default_on_file         str
cb_person_cred_hist_length      int64
date                            int64
year                            int64
month                           int64
unnamed_0                         str
unnamed_1                         str
ipca                          float64
no                            float64
dez_93_100                    float64
no_1                          float64
n12                           float64
ipca_tipo                         str
mes_num                         int64
data_ref                          str
selic_media                   float64
dtype: objec

In [22]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score 
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [23]:
# Features 

features = [
    'person_age', 'person_income', 'person_emp_length', 'loan_amnt',
    'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
    'cb_person_default_on_file'
]

target = 'loan_status'


In [24]:
# Tratamento

df_model_lr = df[features + [target]].copy()

# converter Y/N -> 1/0 
df_model_lr['cb_person_default_on_file'] = df_model_lr['cb_person_default_on_file'].map({'Y': 1, 'N': 0})

# garantir numérico 
for col in df_model_lr.columns:
    df_model_lr[col] = pd.to_numeric(df_model_lr[col], errors = 'coerce')

df_model_lr.dropna(inplace=True)

df_model_lr = df_model_lr[df_model_lr[target].isin([0, 1])]
df_model_lr[target] = df_model_lr[target].astype(int)

print(df_model_lr.shape)
df_model_lr.head()

(28638, 9)


,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,cb_person_default_on_file,loan_status
0,22,59000.0,123.0,35000.0,16.02,0.59,3,1,1
1,21,9600.0,5.0,1000.0,11.14,0.10,2,0,0
2,25,9600.0,1.0,5500.0,12.87,0.57,3,0,1
3,23,65500.0,4.0,35000.0,15.23,0.53,2,0,1
4,24,54400.0,8.0,35000.0,14.27,0.55,4,1,1


In [25]:
# Split

X_lr = df_model_lr[features]
y_lr = df_model_lr[target]

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_lr, y_lr, 
    test_size= 0.2,
    random_state=42, 
    stratify=y_lr
)

In [26]:
# Modelo Pipe


pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=1000, 
        class_weight='balanced', 
        random_state=42
    ))
])

In [27]:
# Validação 

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores= cross_val_score(
    pipeline_lr,
    X_train_lr,
    y_train_lr, 
    cv=cv,
    scoring='roc_auc'
)

print(f"AUC-ROC CV: {scores.mean():.4f} +- {scores.std():.4f}")

AUC-ROC CV: 0.8352 +- 0.0096


In [28]:
# Treino Final 

pipeline_lr.fit(X_train_lr, y_train_lr)
print("deu bom")

deu bom


In [29]:
# Teste

y_pred_lr = pipeline_lr.predict(X_test_lr)
y_proba_lr = pipeline_lr.predict_proba(X_test_lr)[:, 1]

auc_lr = roc_auc_score(y_test_lr, y_proba_lr)

print(f"\nAUC-ROC Test: {auc_lr:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_lr, y_pred_lr))


AUC-ROC Test: 0.8312

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.77      0.84      4487
           1       0.47      0.75      0.58      1241

    accuracy                           0.76      5728
   macro avg       0.69      0.76      0.71      5728
weighted avg       0.82      0.76      0.78      5728



In [30]:

# Matriz de confusão e Resumo 

cm_lr = confusion_matrix(y_test_lr, y_pred_lr)
print("Matriz de confusão - Logistic Regression")
print(cm_lr)

print("Resumo do modelo")
print(f"AUC-ROC CV: {scores.mean():.4f} +- {scores.std():.4f}")
print(f"AUC-ROC Test: {auc_lr:.4f}")

Matriz de confusão - Logistic Regression
[[3444 1043]
 [ 312  929]]
Resumo do modelo
AUC-ROC CV: 0.8352 +- 0.0096
AUC-ROC Test: 0.8312
